# CVPR 2026 MSLR Track 2 — Score Maximizer
### Pretrained EfficientNetV2-S + ConvNeXt-Tiny | K-Fold Ensemble | Mixup/CutMix/SpecAugment | SWA | TTA

**Why the previous submission scored 0.008 (essentially random):**
1. Custom SSM used `cumsum` instead of a real selective scan
2. Trained from scratch on 14.7k samples — impossible for 126 classes
3. 3-loss function (CE+MOSCA+CFD) caused training instability
4. Over-compressed data (256→64 range bins) dropped discriminative info

**This pipeline:** ImageNet-pretrained backbones, label-smoothed CE, aggressive augmentation, 5-fold ensemble + TTA + SWA

In [1]:
# ============================================================
# CELL 1: Environment Setup
# ============================================================
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

pip_install('timm')
pip_install('einops')

import torch
print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    mem = torch.cuda.get_device_properties(0).total_memory
    print(f'VRAM: {mem / 1e9:.1f} GB')

PyTorch 2.9.0+cu126, CUDA: True
GPU: Tesla T4
VRAM: 15.6 GB


In [2]:
# ============================================================
# CELL 2: Imports & Configuration
# ============================================================
import os, csv, time, math, random, warnings, gc
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

warnings.filterwarnings('ignore')

# ---- MODE: Set to 'debug' for quick local test, 'compete' for Kaggle ----
MODE = 'compete'   # <<< CHANGE TO 'compete' BEFORE KAGGLE SUBMISSION

class CFG:
    seed        = 42
    device      = 'cuda' if torch.cuda.is_available() else 'cpu'
    num_classes = 126
    num_workers = 0 if MODE == 'debug' else 2

    # Data
    img_size    = 224
    max_time    = 48      # pad/crop time axis (covers all samples: max observed T=43)

    # Training
    num_folds      = 5 if MODE == 'compete' else 2
    train_folds    = list(range(num_folds))
    label_smoothing = 0.1
    weight_decay   = 0.05
    grad_clip      = 1.0
    use_amp        = True
    warmup_epochs  = 3

    # Augmentation
    mixup_alpha  = 0.4
    cutmix_alpha = 1.0
    mix_prob     = 0.5
    noise_std    = 0.02
    freq_mask    = 30
    time_mask    = 8

    # SWA
    swa_frac     = 0.8    # start SWA at 80% of epochs
    swa_lr       = 1e-5

    # TTA
    use_tta      = True

    # Model zoo: each backbone trained in separate fold loop
    if MODE == 'debug':
        models = [
            dict(name='efficientnet_b0', epochs=3, lr=3e-4, bs=16),
        ]
    else:
        models = [
            dict(name='tf_efficientnetv2_s.in21k_ft_in1k', epochs=45, lr=2e-4, bs=48),
            dict(name='convnext_tiny.fb_in22k_ft_in1k',    epochs=45, lr=3e-4, bs=48),
        ]

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.benchmark = True

set_seed(CFG.seed)
print(f'MODE={MODE}, device={CFG.device}, folds={CFG.num_folds}')
print(f'Models: {[m["name"] for m in CFG.models]}')

MODE=compete, device=cuda, folds=5
Models: ['tf_efficientnetv2_s.in21k_ft_in1k', 'convnext_tiny.fb_in22k_ft_in1k']


In [3]:
# ============================================================
# CELL 3: Auto-detect data paths
# ============================================================
def find_data():
    for p in [
        Path('/kaggle/input/2st-multimodal-italian-sign-language-rec'),
        Path('/kaggle/input/cvpr-mslr-2026-track-2'),
        Path('/kaggle/input/competitions/cvpr-mslr-2026-track-2'),
        Path('d:/Current-Research/CVPR2026-SignEval/cvpr-mslr-2026-track-2'),
        Path('../cvpr-mslr-2026-track-2'),
    ]:
        if (p / 'train').exists():
            return p
    raise FileNotFoundError('Competition dataset not found! Attach it in Kaggle Input.')

DATA_ROOT = find_data()
TRAIN_DIR = DATA_ROOT / 'train'
VAL_DIR   = DATA_ROOT / 'val'
OUT       = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('./output')
OUT.mkdir(parents=True, exist_ok=True)

print(f'Data root: {DATA_ROOT}')
print(f'Train dir: {TRAIN_DIR}')
print(f'Val dir  : {VAL_DIR}')
print(f'Output   : {OUT}')

Data root: /kaggle/input/competitions/cvpr-mslr-2026-track-2
Train dir: /kaggle/input/competitions/cvpr-mslr-2026-track-2/train
Val dir  : /kaggle/input/competitions/cvpr-mslr-2026-track-2/val
Output   : /kaggle/working


In [4]:
# ============================================================
# CELL 4: Index all samples + Create folds + Test set
# ============================================================
def build_train_samples(train_dir):
    """Return list of (sample_dir_path, label_int) + class_names list."""
    samples = []
    class_dirs = sorted(
        [d for d in Path(train_dir).iterdir()
         if d.is_dir() and d.name.split('_')[0].isdigit()],
        key=lambda d: int(d.name.split('_')[0]))
    class_names = [d.name for d in class_dirs]
    for cd in class_dirs:
        label = int(cd.name.split('_')[0])
        for sd in sorted(cd.iterdir()):
            if sd.is_dir() and sd.name.startswith('SAMPLE_'):
                if (sd / f'{sd.name}_RTM1.npy').exists():
                    samples.append((str(sd), label))
    return samples, class_names

def build_folds(samples, n_folds, seed=42):
    """Stratified k-fold: each class is evenly distributed."""
    rng = np.random.RandomState(seed)
    c2i = defaultdict(list)
    for i, (_, l) in enumerate(samples):
        c2i[l].append(i)
    folds = [[] for _ in range(n_folds)]
    for cls in sorted(c2i):
        idxs = c2i[cls].copy()
        rng.shuffle(idxs)
        for i, idx in enumerate(idxs):
            folds[i % n_folds].append(idx)
    return folds

def build_test_samples(val_dir):
    """Return list of (sample_dir_path, 'SAMPLE_ID') for submission."""
    samples = []
    for d in sorted(Path(val_dir).iterdir()):
        if d.is_dir() and d.name.startswith('SAMPLE_'):
            if (d / f'{d.name}_RTM1.npy').exists():
                samples.append((str(d), d.name))
    return samples

all_train_samples, class_names = build_train_samples(TRAIN_DIR)
fold_indices = build_folds(all_train_samples, CFG.num_folds, CFG.seed)
test_samples = build_test_samples(VAL_DIR)

print(f'Training: {len(all_train_samples)} samples, {len(class_names)} classes')
print(f'Test:     {len(test_samples)} samples (unlabeled)')
print(f'Folds:    {[len(f) for f in fold_indices]}')
print(f'Classes:  {class_names[:5]} ... {class_names[-3:]}')

# Sanity: check a sample
sd0, lbl0 = all_train_samples[0]
sid0 = Path(sd0).name
arr0 = np.load(os.path.join(sd0, f'{sid0}_RTM1.npy'))
print(f'\nSample: {sid0}, label={lbl0}, RTM1 shape={arr0.shape}, '
      f'range=[{arr0.min():.1f}, {arr0.max():.1f}]')

Training: 14742 samples, 126 classes
Test:     4914 samples (unlabeled)
Folds:    [3024, 3024, 2898, 2898, 2898]
Classes:  ['0_A', '1_AIUTARE', '2_ALTRO', '3_AMBULANZA', '4_ANSIA'] ... ['123_X', '124_Y', '125_Z']

Sample: SAMPLE_10462, label=0, RTM1 shape=(16, 256), range=[-108.8, -21.8]


In [5]:
# ============================================================
# CELL 5: Dataset class
# ============================================================
class RTMDataset(Dataset):
    """
    Load 3-antenna RTM .npy files as 3-channel spectrogram images.
    Pipeline: load(3,256,T) -> normalize -> pad_time -> resize(224,224) -> augment
    """
    def __init__(self, sample_list, img_size=224, max_T=48, augment=False):
        self.samples  = sample_list
        self.img_size = img_size
        self.max_T    = max_T
        self.augment  = augment

    def __len__(self):
        return len(self.samples)

    def _load(self, sample_dir):
        sd = Path(sample_dir)
        sid = sd.name
        rtms = []
        for i in range(1, 4):
            arr = np.load(str(sd / f'{sid}_RTM{i}.npy')).astype(np.float32)
            rtms.append(arr)  # each (T, 256)
        stacked = np.stack(rtms, axis=0)  # (3, T, 256)
        return stacked.transpose(0, 2, 1)  # (3, 256, T)

    def _normalize(self, x):
        mn, mx = x.min(), x.max()
        return (x - mn) / (mx - mn + 1e-8)

    def _pad_time(self, rtm):
        C, H, T = rtm.shape
        if T >= self.max_T:
            s = (T - self.max_T) // 2
            return rtm[:, :, s:s + self.max_T]
        pad = self.max_T - T
        pl, pr = pad // 2, pad - pad // 2
        return np.pad(rtm, ((0,0), (0,0), (pl, pr)), mode='constant')

    def _resize(self, t):
        return F.interpolate(
            t.unsqueeze(0),
            size=(self.img_size, self.img_size),
            mode='bilinear', align_corners=False
        ).squeeze(0)

    def _augment_np(self, rtm):
        # 1) Random time flip
        if random.random() < 0.3:
            rtm = rtm[:, :, ::-1].copy()
        # 2) Small time shift
        shift = random.randint(-3, 3)
        if shift:
            rtm = np.roll(rtm, shift, axis=2)
        # 3) Gaussian noise
        if random.random() < 0.5:
            rtm = np.clip(
                rtm + np.random.randn(*rtm.shape).astype(np.float32) * CFG.noise_std,
                0, 1)
        return rtm

    def _spec_augment(self, img):
        C, H, W = img.shape
        # Freq mask
        if random.random() < 0.5:
            f = random.randint(1, CFG.freq_mask)
            f0 = random.randint(0, max(0, H - f))
            img[:, f0:f0+f, :] = 0
        # Time mask
        if random.random() < 0.5:
            t = random.randint(1, CFG.time_mask)
            t0 = random.randint(0, max(0, W - t))
            img[:, :, t0:t0+t] = 0
        # Second freq mask (smaller)
        if random.random() < 0.3:
            f = random.randint(1, CFG.freq_mask // 2)
            f0 = random.randint(0, max(0, H - f))
            img[:, f0:f0+f, :] = 0
        return img

    def __getitem__(self, idx):
        sd, label_or_id = self.samples[idx]
        rtm = self._load(sd)             # (3, 256, T)
        rtm = self._normalize(rtm)       # [0, 1]
        rtm = self._pad_time(rtm)        # (3, 256, max_T)
        if self.augment:
            rtm = self._augment_np(rtm)
        img = self._resize(torch.from_numpy(rtm.copy()).float())   # (3, 224, 224)
        if self.augment:
            img = self._spec_augment(img)
        return img, label_or_id

# --- Quick sanity check ---
_ds = RTMDataset(all_train_samples[:4], img_size=CFG.img_size, max_T=CFG.max_time, augment=True)
img0, lbl0 = _ds[0]
print(f'Dataset output shape: {img0.shape}, dtype: {img0.dtype}, range: [{img0.min():.3f}, {img0.max():.3f}]')
print(f'Label: {lbl0}')
assert img0.shape == (3, 224, 224), f'Wrong shape: {img0.shape}'
assert img0.min() >= 0 and img0.max() <= 1.01, f'Values out of range'
print('Dataset sanity check PASSED')
del _ds

Dataset output shape: torch.Size([3, 224, 224]), dtype: torch.float32, range: [0.000, 0.998]
Label: 0
Dataset sanity check PASSED


In [6]:
# ============================================================
# CELL 6: Mixup / CutMix utilities
# ============================================================
def mixup_data(x, y, alpha=0.4):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def cutmix_data(x, y, alpha=1.0):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    _, _, H, W = x.shape
    r = np.sqrt(1 - lam)
    ch, cw = int(H * r), int(W * r)
    cy = random.randint(0, H)
    cx = random.randint(0, W)
    y1, y2 = max(0, cy - ch//2), min(H, cy + ch//2)
    x1, x2 = max(0, cx - cw//2), min(W, cx + cw//2)
    xc = x.clone()
    xc[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam = 1 - (y2-y1) * (x2-x1) / (H * W)
    return xc, y, y[idx], lam

def mix_criterion(crit, pred, ya, yb, lam):
    return lam * crit(pred, ya) + (1 - lam) * crit(pred, yb)

print('Mixup / CutMix ready')

Mixup / CutMix ready


In [7]:
# ============================================================
# CELL 7: Model builder + quick test
# ============================================================
def build_model(name, num_classes=126, pretrained=True):
    m = timm.create_model(
        name,
        pretrained=pretrained,
        num_classes=num_classes,
        in_chans=3,
        drop_rate=0.3,
        drop_path_rate=0.2,
    )
    n = sum(p.numel() for p in m.parameters()) / 1e6
    print(f'  Built {name}: {n:.1f}M params, pretrained={pretrained}')
    return m

# Quick forward-pass test
_m = build_model(CFG.models[0]['name']).to(CFG.device)
_x = torch.randn(2, 3, 224, 224).to(CFG.device)
with torch.no_grad():
    _out = _m(_x)
print(f'  Forward test: input={_x.shape} -> output={_out.shape}')
assert _out.shape == (2, 126), f'Wrong output shape: {_out.shape}'
print('  Model forward pass PASSED')
del _m, _x, _out
torch.cuda.empty_cache()

model.safetensors:   0%|          | 0.00/86.5M [00:00<?, ?B/s]

  Built tf_efficientnetv2_s.in21k_ft_in1k: 20.3M params, pretrained=True
  Forward test: input=torch.Size([2, 3, 224, 224]) -> output=torch.Size([2, 126])
  Model forward pass PASSED


In [8]:
# ============================================================
# CELL 8: LR scheduler + Train / Validate functions
# ============================================================
def cosine_lr(optimizer, warmup, total, min_frac=0.01):
    def fn(ep):
        if ep < warmup:
            return (ep + 1) / warmup
        prog = (ep - warmup) / max(1, total - warmup)
        return min_frac + 0.5 * (1 - min_frac) * (1 + math.cos(math.pi * prog))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, fn)


def train_one_epoch(model, loader, opt, sched, scaler, crit, epoch, total_ep):
    model.train()
    loss_sum = 0.0
    correct = total = 0

    for step, (imgs, labels) in enumerate(loader):
        imgs   = imgs.to(CFG.device, non_blocking=True)
        labels = labels.to(CFG.device, non_blocking=True)

        # Mixup or CutMix
        do_mix = random.random() < CFG.mix_prob and epoch < total_ep - 3
        if do_mix:
            if random.random() < 0.5:
                imgs, ya, yb, lam = mixup_data(imgs, labels, CFG.mixup_alpha)
            else:
                imgs, ya, yb, lam = cutmix_data(imgs, labels, CFG.cutmix_alpha)

        with autocast(enabled=CFG.use_amp):
            logits = model(imgs)
            if do_mix:
                loss = mix_criterion(crit, logits, ya, yb, lam)
            else:
                loss = crit(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
        scaler.step(opt)
        scaler.update()
        opt.zero_grad(set_to_none=True)

        loss_sum += loss.item()
        if not do_mix:
            correct += (logits.argmax(1) == labels).sum().item()
            total   += labels.size(0)

    sched.step()
    return loss_sum / max(len(loader), 1), correct / max(total, 1) * 100


@torch.no_grad()
def validate(model, loader):
    model.eval()
    logits_all, labels_all = [], []
    for imgs, labels in loader:
        imgs = imgs.to(CFG.device, non_blocking=True)
        with autocast(enabled=CFG.use_amp):
            logits_all.append(model(imgs).cpu())
        labels_all.append(labels)
    logits_all = torch.cat(logits_all)
    labels_all = torch.cat(labels_all)
    top1 = (logits_all.argmax(1) == labels_all).float().mean().item() * 100
    _, t5 = logits_all.topk(5, 1)
    top5 = t5.eq(labels_all.unsqueeze(1)).any(1).float().mean().item() * 100
    return top1, top5

print('Training functions ready')

Training functions ready


In [9]:
# ============================================================
# CELL 9: Quick 1-epoch debug test (only runs in debug mode)
# ============================================================
if MODE == 'debug':
    print('=== DEBUG: 1-epoch smoke test ===')
    _train_ds = RTMDataset(
        all_train_samples[:64],
        img_size=CFG.img_size, max_T=CFG.max_time, augment=True)
    _val_ds = RTMDataset(
        all_train_samples[64:96],
        img_size=CFG.img_size, max_T=CFG.max_time, augment=False)
    _tl = DataLoader(_train_ds, batch_size=8, shuffle=True, num_workers=0, drop_last=True)
    _vl = DataLoader(_val_ds, batch_size=8, shuffle=False, num_workers=0)
    
    _model = build_model(CFG.models[0]['name']).to(CFG.device)
    _opt   = torch.optim.AdamW(_model.parameters(), lr=1e-3, weight_decay=0.01)
    _sched = cosine_lr(_opt, 1, 3)
    _scaler = GradScaler(enabled=CFG.use_amp)
    _crit  = nn.CrossEntropyLoss(label_smoothing=CFG.label_smoothing)
    
    t0 = time.time()
    loss, acc = train_one_epoch(_model, _tl, _opt, _sched, _scaler, _crit, 0, 3)
    v1, v5   = validate(_model, _vl)
    dt = time.time() - t0
    
    print(f'  Train loss={loss:.3f}, acc={acc:.1f}%')
    print(f'  Val top1={v1:.1f}%, top5={v5:.1f}%')
    print(f'  Time: {dt:.1f}s')
    print(f'  GPU memory: {torch.cuda.memory_allocated()/1e6:.0f} MB' if torch.cuda.is_available() else '')
    print('=== DEBUG: Smoke test PASSED ===')
    
    del _model, _opt, _sched, _scaler, _crit, _tl, _vl, _train_ds, _val_ds
    gc.collect()
    torch.cuda.empty_cache()
else:
    print('Skipping debug smoke test (MODE=compete)')

Skipping debug smoke test (MODE=compete)


In [10]:
# ============================================================
# CELL 10: Full K-Fold Training Loop (all models)
# ============================================================
all_model_paths = []
all_fold_accs   = []
TOTAL_START     = time.time()

for mcfg in CFG.models:
    mname  = mcfg['name']
    epochs = mcfg['epochs']
    lr     = mcfg['lr']
    bs     = mcfg['bs']
    swa_start = int(epochs * CFG.swa_frac)

    print(f'\n{"="*70}')
    print(f'MODEL: {mname} | epochs={epochs} | lr={lr} | bs={bs} | SWA@{swa_start}')
    print(f'{"="*70}')

    for fold in CFG.train_folds:
        fold_start = time.time()
        print(f'\n--- Fold {fold}/{CFG.num_folds-1} ---')

        # Split indices
        val_idx   = fold_indices[fold]
        train_idx = [i for f in range(CFG.num_folds) if f != fold for i in fold_indices[f]]
        train_samples = [all_train_samples[i] for i in train_idx]
        val_samples   = [all_train_samples[i] for i in val_idx]
        print(f'  Train: {len(train_samples)}, Val: {len(val_samples)}')

        # DataLoaders
        train_ds = RTMDataset(train_samples, img_size=CFG.img_size, max_T=CFG.max_time, augment=True)
        val_ds   = RTMDataset(val_samples,   img_size=CFG.img_size, max_T=CFG.max_time, augment=False)
        train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True,
                                  num_workers=CFG.num_workers, pin_memory=True, drop_last=True)
        val_loader   = DataLoader(val_ds,   batch_size=bs*2, shuffle=False,
                                  num_workers=CFG.num_workers, pin_memory=True)

        # Model + optimizer
        model   = build_model(mname).to(CFG.device)
        opt     = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=CFG.weight_decay)
        sched   = cosine_lr(opt, CFG.warmup_epochs, epochs)
        scaler  = GradScaler(enabled=CFG.use_amp)
        crit    = nn.CrossEntropyLoss(label_smoothing=CFG.label_smoothing)

        # SWA model
        swa_model = torch.optim.swa_utils.AveragedModel(model)
        swa_sched = torch.optim.swa_utils.SWALR(opt, swa_lr=CFG.swa_lr)

        best_acc  = 0.0
        best_path = OUT / f'best_{mname.replace("/","_").replace(".","_")}_f{fold}.pt'

        for ep in range(epochs):
            t0 = time.time()
            loss, tacc = train_one_epoch(model, train_loader, opt, sched, scaler, crit, ep, epochs)

            # SWA update
            if ep >= swa_start:
                swa_model.update_parameters(model)
                swa_sched.step()

            val1, val5 = validate(model, val_loader)
            dt = time.time() - t0

            # Log every 5 epochs or on improvement
            if (ep+1) % 5 == 0 or val1 > best_acc:
                print(f'  Ep {ep:3d} | loss={loss:.3f} | '
                      f'val={val1:.1f}%/{val5:.1f}% | '
                      f'lr={opt.param_groups[0]["lr"]:.1e} | {dt:.0f}s')

            if val1 > best_acc:
                best_acc = val1
                torch.save({
                    'model': model.state_dict(),
                    'name':  mname,
                    'acc':   val1,
                    'fold':  fold,
                    'epoch': ep,
                    'img_size': CFG.img_size,
                }, best_path)
                print(f'    >>> New best: {best_acc:.2f}%')

        # SWA finalize
        try:
            torch.optim.swa_utils.update_bn(train_loader, swa_model, device=CFG.device)
            swa_acc, _ = validate(swa_model, val_loader)
            print(f'  SWA val: {swa_acc:.1f}% (best checkpoint: {best_acc:.1f}%)')
            if swa_acc > best_acc:
                best_acc = swa_acc
                torch.save({
                    'model': swa_model.module.state_dict(),
                    'name':  mname,
                    'acc':   swa_acc,
                    'fold':  fold,
                    'epoch': epochs,
                    'img_size': CFG.img_size,
                }, best_path)
                print(f'    >>> SWA is better! Saved.')
        except Exception as e:
            print(f'  SWA BN update failed: {e}  (using best checkpoint)')

        all_model_paths.append(str(best_path))
        all_fold_accs.append(best_acc)
        fold_time = time.time() - fold_start
        print(f'  Fold {fold} done: best={best_acc:.2f}%, time={fold_time/60:.1f}min')

        del model, swa_model, opt, sched, scaler
        gc.collect()
        torch.cuda.empty_cache()

total_time = time.time() - TOTAL_START
print(f'\n{"="*70}')
print(f'ALL TRAINING COMPLETE in {total_time/3600:.1f} hours')
print(f'Fold accs: {["{:.1f}%".format(a) for a in all_fold_accs]}')
print(f'Mean: {np.mean(all_fold_accs):.2f}% +/- {np.std(all_fold_accs):.2f}%')
print(f'Checkpoints: {all_model_paths}')
print(f'{"="*70}')


MODEL: tf_efficientnetv2_s.in21k_ft_in1k | epochs=45 | lr=0.0002 | bs=48 | SWA@36

--- Fold 0/4 ---
  Train: 11718, Val: 3024
  Built tf_efficientnetv2_s.in21k_ft_in1k: 20.3M params, pretrained=True
  Ep   0 | loss=5.016 | val=1.7%/9.1% | lr=1.3e-04 | 227s
    >>> New best: 1.72%
  Ep   1 | loss=4.644 | val=3.5%/15.8% | lr=2.0e-04 | 74s
    >>> New best: 3.47%
  Ep   2 | loss=4.293 | val=17.5%/49.0% | lr=2.0e-04 | 74s
    >>> New best: 17.49%
  Ep   3 | loss=3.703 | val=34.9%/71.9% | lr=2.0e-04 | 74s
    >>> New best: 34.92%
  Ep   4 | loss=3.273 | val=52.5%/84.1% | lr=2.0e-04 | 74s
    >>> New best: 52.51%
  Ep   5 | loss=3.042 | val=55.4%/86.6% | lr=2.0e-04 | 74s
    >>> New best: 55.39%
  Ep   6 | loss=2.786 | val=64.1%/91.4% | lr=2.0e-04 | 74s
    >>> New best: 64.05%
  Ep   7 | loss=2.585 | val=64.8%/90.9% | lr=1.9e-04 | 74s
    >>> New best: 64.85%
  Ep   8 | loss=2.476 | val=69.5%/93.0% | lr=1.9e-04 | 74s
    >>> New best: 69.51%
  Ep   9 | loss=2.301 | val=71.9%/93.8% | lr=1.9

model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

  Built convnext_tiny.fb_in22k_ft_in1k: 27.9M params, pretrained=True
  Ep   0 | loss=4.869 | val=0.7%/4.9% | lr=2.0e-04 | 102s
    >>> New best: 0.66%
  Ep   1 | loss=4.732 | val=2.0%/10.1% | lr=3.0e-04 | 67s
    >>> New best: 1.95%
  Ep   2 | loss=4.613 | val=2.4%/10.8% | lr=3.0e-04 | 67s
    >>> New best: 2.45%
  Ep   4 | loss=4.486 | val=3.4%/15.6% | lr=3.0e-04 | 67s
    >>> New best: 3.44%
  Ep   5 | loss=4.429 | val=4.1%/16.7% | lr=3.0e-04 | 67s
    >>> New best: 4.10%
  Ep   6 | loss=4.352 | val=5.5%/22.9% | lr=2.9e-04 | 67s
    >>> New best: 5.52%
  Ep   7 | loss=4.190 | val=8.7%/31.6% | lr=2.9e-04 | 67s
    >>> New best: 8.73%
  Ep   8 | loss=3.929 | val=20.1%/56.2% | lr=2.9e-04 | 67s
    >>> New best: 20.07%
  Ep   9 | loss=3.576 | val=34.6%/72.8% | lr=2.8e-04 | 68s
    >>> New best: 34.56%
  Ep  10 | loss=3.389 | val=42.8%/79.8% | lr=2.7e-04 | 67s
    >>> New best: 42.79%
  Ep  11 | loss=3.104 | val=52.3%/85.4% | lr=2.7e-04 | 67s
    >>> New best: 52.31%
  Ep  12 | loss=2.78

In [11]:
# ============================================================
# CELL 11: Ensemble Inference with TTA
# ============================================================
@torch.no_grad()
def predict_tta(model, loader, use_tta=True):
    model.eval()
    probs_list, ids_list = [], []

    for imgs, sids in loader:
        imgs = imgs.to(CFG.device, non_blocking=True)

        with autocast(enabled=CFG.use_amp):
            p = F.softmax(model(imgs), dim=1)

        if use_tta:
            # TTA 1: horizontal flip (time-reverse)
            with autocast(enabled=CFG.use_amp):
                p_flip = F.softmax(model(torch.flip(imgs, [3])), dim=1)
            p = (p + p_flip) / 2.0

        probs_list.append(p.cpu())
        ids_list.extend(sids if isinstance(sids[0], str) else sids.tolist())

    return torch.cat(probs_list), ids_list


print(f'Generating ensemble predictions from {len(all_model_paths)} models...')

test_ds     = RTMDataset(test_samples, img_size=CFG.img_size, max_T=CFG.max_time, augment=False)
test_loader = DataLoader(test_ds, batch_size=96, shuffle=False,
                         num_workers=CFG.num_workers, pin_memory=True)

ensemble_probs = None
sample_ids     = None

for i, path in enumerate(all_model_paths):
    ckpt = torch.load(path, map_location='cpu', weights_only=False)
    mname = ckpt['name']
    acc   = ckpt.get('acc', 0)
    fold  = ckpt.get('fold', '?')
    print(f'  [{i+1}/{len(all_model_paths)}] {mname} fold={fold} val={acc:.1f}%')

    model = timm.create_model(mname, pretrained=False, num_classes=CFG.num_classes,
                              drop_rate=0, drop_path_rate=0)
    model.load_state_dict(ckpt['model'])
    model = model.to(CFG.device)

    probs, ids = predict_tta(model, test_loader, use_tta=CFG.use_tta)

    if ensemble_probs is None:
        ensemble_probs = probs
        sample_ids     = ids
    else:
        ensemble_probs += probs

    del model, ckpt
    gc.collect()
    torch.cuda.empty_cache()

ensemble_probs /= len(all_model_paths)
preds = ensemble_probs.argmax(dim=1).numpy()

print(f'\nPredictions: {len(preds)} samples')
print(f'Unique classes predicted: {len(np.unique(preds))} / {CFG.num_classes}')
print(f'Prediction distribution (first 10): {np.bincount(preds, minlength=126)[:10]}')

Generating ensemble predictions from 10 models...
  [1/10] tf_efficientnetv2_s.in21k_ft_in1k fold=0 val=82.1%
  [2/10] tf_efficientnetv2_s.in21k_ft_in1k fold=1 val=82.8%
  [3/10] tf_efficientnetv2_s.in21k_ft_in1k fold=2 val=82.2%
  [4/10] tf_efficientnetv2_s.in21k_ft_in1k fold=3 val=81.6%
  [5/10] tf_efficientnetv2_s.in21k_ft_in1k fold=4 val=82.6%
  [6/10] convnext_tiny.fb_in22k_ft_in1k fold=0 val=83.0%
  [7/10] convnext_tiny.fb_in22k_ft_in1k fold=1 val=83.4%
  [8/10] convnext_tiny.fb_in22k_ft_in1k fold=2 val=84.1%
  [9/10] convnext_tiny.fb_in22k_ft_in1k fold=3 val=79.4%
  [10/10] convnext_tiny.fb_in22k_ft_in1k fold=4 val=83.8%

Predictions: 4914 samples
Unique classes predicted: 126 / 126
Prediction distribution (first 10): [37 39 43 38 39 34 38 42 38 42]


In [12]:
# ============================================================
# CELL 12: Write submission CSV
# ============================================================
rows = []
for sid, pred in zip(sample_ids, preds):
    nid = int(sid.replace('SAMPLE_', '')) if isinstance(sid, str) else int(sid)
    rows.append({'id': nid, 'Pred': int(pred)})

rows.sort(key=lambda r: r['id'])

sub_path = OUT / 'submission.csv'
with open(sub_path, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['id', 'Pred'])
    w.writeheader()
    w.writerows(rows)

print(f'Submission saved: {sub_path}')
print(f'Total rows: {len(rows)}')

# Sanity checks
import pandas as pd
df = pd.read_csv(sub_path)
print(f'\n--- Sanity Check ---')
print(f'Shape:       {df.shape}')
print(f'Pred range:  [{df.Pred.min()}, {df.Pred.max()}]')
print(f'Unique Pred: {df.Pred.nunique()}')
print(f'Any NaN:     {df.isnull().any().any()}')
print(f'\nHead:')
print(df.head(10))
print(f'\nTail:')
print(df.tail(5))
print(f'\nClass distribution (top 10):')
print(df.Pred.value_counts().head(10))

Submission saved: /kaggle/working/submission.csv
Total rows: 4914

--- Sanity Check ---
Shape:       (4914, 2)
Pred range:  [0, 125]
Unique Pred: 126
Any NaN:     False

Head:
   id  Pred
0   1    72
1   4     9
2   6   101
3  12    39
4  17    19
5  20    24
6  21    17
7  22    52
8  25     2
9  26     3

Tail:
         id  Pred
4909  24558    29
4910  24560    70
4911  24561    46
4912  24564   102
4913  24566    73

Class distribution (top 10):
Pred
51     60
52     54
46     50
16     48
113    47
40     46
97     46
45     46
76     45
108    44
Name: count, dtype: int64


---
## Done!

### Expected score ranges:
- **Single EfficientNetV2-S fold:** ~0.70-0.80
- **5-fold EfficientNetV2-S ensemble + TTA:** ~0.80-0.85
- **Multi-backbone 10-model ensemble + TTA + SWA:** ~0.85-0.90

### To boost further:
1. Add more backbones: `swin_base_patch4_window7_224`, `vit_base_patch16_224`
2. Increase image resolution to 384 (with smaller batch size)
3. Use pseudo-RDM (FFT along time) as additional 3 channels (6-ch input)
4. Heavier augmentation: RandAugment, random erasing
5. Knowledge distillation from larger teacher model